In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    roc_curve,
    roc_auc_score
)

import statsmodels.api as sm


In [2]:
#importamos el dataset  y el preprocesamiento
base_path = Path.cwd()

X_train = pd.read_csv(base_path / "X_train_impago.csv")
X_test = pd.read_csv(base_path / "X_test_impago.csv")

y_train = pd.read_csv(base_path / "y_train_impago.csv").squeeze("columns")
y_test = pd.read_csv(base_path / "y_test_impago.csv").squeeze("columns")

In [3]:
#comprobamos la distribucion de impago
print("\nPorcentaje de observaciones por clase en train:")
print(y_train.value_counts(normalize=True) * 100)

print("\nPorcentaje de observaciones por clase en test:")
print(y_test.value_counts(normalize=True) * 100)



Porcentaje de observaciones por clase en train:
Impago
0    88.447976
1    11.552024
Name: proportion, dtype: float64

Porcentaje de observaciones por clase en test:
Impago
0    88.449789
1    11.550211
Name: proportion, dtype: float64


In [4]:
#hacemos ajustes del modelo con sklearn

modelo_log_sklearn = LogisticRegression()

modelo_log_sklearn.fit(X_train, y_train)


# Información del modelo

print("\nIntercepto:")
print(modelo_log_sklearn.intercept_)

coeficientes = pd.DataFrame({
    "Variable": X_train.columns,
    "Coeficiente": modelo_log_sklearn.coef_.flatten()
})

print("\nCoeficientes:")
print(coeficientes)


print(
    "\nAccuracy de entrenamiento:",
    modelo_log_sklearn.score(X_train, y_train))


Intercepto:
[-2.33601145]

Coeficientes:
                        Variable  Coeficiente
0                          Prima     0.005377
1             Scoring_Crediticio    -0.121507
2                   Meses_Empleo    -0.269188
3                       Ingresos     0.031602
4           Ratio_Deuda_Ingresos    -0.053486
5                  Monto_Inicial    -0.023371
6                   Num_Creditos    -0.000396
7                       Duracion    -0.023597
8                           Edad    -0.341578
9                  Ratio_Interes     0.411211
10              Estudios_Escolar     0.151676
11  Estudios_Grado Universitario     0.078845
12               Estudios_Máster     0.013495
13       Estado_Civil_Divorciado    -0.019647
14          Estado_Civil_Soltero     0.007270
15         Posesion_Hipoteca_1.0     0.048691

Accuracy de entrenamiento: 0.8843985115130243


Realizamos predicciones probabilísticas:

- Con .predict_proba() se obtiene, para cada observación, la probabilidad predicha de pertenecer a cada una de las dos clases.

- Con .predict() se obtiene, para cada observación, la clasificación predicha por el modelo. Esta clasificación se corresponde con la clase con mayor probabilidad.

In [5]:
#hacemos las predicciones de que pertenezca a una clase o a otra (pago/impago)
probabilidades = modelo_log_sklearn.predict_proba(X_test)

probabilidades = pd.DataFrame(
    probabilidades,
    columns=modelo_log_sklearn.classes_
)

print("\nPrimeras probabilidades:")
print(probabilidades.head())


# Nos quedamos con la probabilidad de Impago = 1
prob_impago = probabilidades[1]


# Clasificación utilizando threshold = 0.5
clasificacion = np.where(prob_impago < 0.5, 0, 1)

print("\nPrimeras clasificaciones:")
print(clasificacion[:10])


Primeras probabilidades:
          0         1
0  0.925994  0.074006
1  0.780709  0.219291
2  0.922595  0.077405
3  0.864693  0.135307
4  0.847288  0.152712

Primeras clasificaciones:
[0 0 0 0 0 0 0 0 0 0]


In [8]:
#hacemos el modelo
# Statsmodels necesita que todas las variables sean numéricas
X_train_stats = X_train.astype(float)

# Añadir constante para calcular el intercepto
X_train_stats = sm.add_constant(X_train_stats, prepend=True)

modelo_log_statsm = sm.Logit(
    endog=y_train,
    exog=X_train_stats
)

modelo_log_statsm = modelo_log_statsm.fit()

print("\nResumen del modelo:")
print(modelo_log_statsm.summary())

Optimization terminated successfully.
         Current function value: 0.333540
         Iterations 7

Resumen del modelo:
                           Logit Regression Results                           
Dep. Variable:                 Impago   No. Observations:                61539
Model:                          Logit   Df Residuals:                    61522
Method:                           MLE   Df Model:                           16
Date:                Mon, 05 Oct 2026   Pseudo R-squ.:                 0.06807
Time:                        16:28:41   Log-Likelihood:                -20526.
converged:                       True   LL-Null:                       -22025.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                           -2.3346      0.071 

In [9]:
#intervalos de confianza de los coeficientes
intervalos_ci = modelo_log_statsm.conf_int(alpha=0.05).rename(columns={0: "2.5%", 1: "97.5%"})

intervalos_ci[:4]

,2.5%,97.5%
const,-2.473328,-2.195849
Prima,-0.039777,0.050256
Scoring_Crediticio,-0.148783,-0.094664
Meses_Empleo,-0.389982,-0.153239


METRICAS DE RENDIMIENTO


In [ ]:
# Accuracy / Aciertos: 
""" 
Mide la proporción de predicciones correctas sobre el total de observaciones evaluadas
"""

from sklearn.metrics import accuracy_score

acierto = accuracy_score(y_test, clasificacion)
error = 1 - acierto
print("Acierto:", round(acierto*100, 2), "%")
print("Error:", round(error*100, 2), "%")Q

Acierto: 88.43 %
Error: 11.57 %


In [11]:
# Precisión, Recall, f1-score
""" 
Precision: Cada vez que el modelo predijo una clase, acerto ese porcentaje
Recall: De todos los datos que eran de esa clase, detectó ese porcentaje
F1-score: El valor será alto si precisión y sensibilidad son similares, útil para comparar clasificadores.
"""



from sklearn.metrics import classification_report

print(classification_report(y_test, clasificacion))

              precision    recall  f1-score   support

           0       0.88      1.00      0.94     13608
           1       0.20      0.00      0.00      1777

    accuracy                           0.88     15385
   macro avg       0.54      0.50      0.47     15385
weighted avg       0.81      0.88      0.83     15385

